In [2]:
import os
import groq
import pandas as pd
import numpy as np
import joblib
import re
import warnings

warnings.filterwarnings('ignore')

# Verify API key is loaded
api_key = os.environ.get('GROQ_API_KEY')

if api_key:
    print(f"Groq API key loaded successfully")
    print(f"Key preview: {api_key[:8]}...")
else:
    print("ERROR: API key not found. Check environment variable setup.")

print("\nLibraries loaded successfully")

Groq API key loaded successfully
Key preview: gsk_Cvep...

Libraries loaded successfully


In [3]:
# Load everything needed for the AI feature
print("Loading models and data...")

# Load tuned model and SHAP explainer
tuned_model = joblib.load('../src/model_tuned_lightgbm.pkl')
shap_explainer = joblib.load('../src/shap_explainer.pkl')
shap_importance = joblib.load('../src/shap_importance.pkl')

# Load a small sample of processed data
X = pd.read_csv('../data/X_processed.csv')
y = pd.read_csv('../data/y_target.csv').squeeze()

# Clean column names
def clean_column_names(df):
    df.columns = [re.sub(r'[^A-Za-z0-9_]', '_', col) for col in df.columns]
    return df

X = clean_column_names(X)

# Sample 10 applicants for demonstration
X_demo = X.sample(10, random_state=42)
y_demo = y[X_demo.index]

print(f"Model loaded: {type(tuned_model).__name__}")
print(f"Demo applicants: {len(X_demo)}")
print(f"Top 5 features by importance:")
print(shap_importance.head(5)[['feature', 'importance']].to_string(index=False))

Loading models and data...
Model loaded: LGBMClassifier
Demo applicants: 10
Top 5 features by importance:
        feature  importance
EXT_SOURCE_MEAN    0.365917
   EXT_SOURCE_3    0.194652
      LOAN_TERM    0.121793
   EXT_SOURCE_2    0.106347
 EXT_SOURCE_MIN    0.095095


In [5]:
# Function to extract SHAP explanations for a single applicant
def get_applicant_shap_explanation(applicant_row, n_top=5):
    """
    Computes SHAP values for a single applicant and returns
    the top risk factors and mitigating factors.
    """
    # Compute SHAP values for this applicant
    shap_vals = shap_explainer.shap_values(applicant_row)
    
    # Handle list output from LightGBM
    if isinstance(shap_vals, list):
        shap_vals = shap_vals[1]
    
    # Build a dataframe of feature contributions
    shap_df = pd.DataFrame({
        'feature': applicant_row.columns,
        'shap_value': shap_vals[0],
        'feature_value': applicant_row.values[0]
    }).sort_values('shap_value', ascending=False)
    
    # Top risk factors — pushing towards default
    risk_factors = shap_df[shap_df['shap_value'] > 0].head(n_top)
    
    # Top mitigating factors — pushing away from default
    mitigating_factors = shap_df[shap_df['shap_value'] < 0].tail(n_top)
    mitigating_factors = mitigating_factors.sort_values('shap_value', ascending=True)
    
    return risk_factors, mitigating_factors

# Test it on the first demo applicant
test_applicant = X_demo.iloc[[0]]
prob = tuned_model.predict_proba(test_applicant)[0][1]

risk_factors, mitigating_factors = get_applicant_shap_explanation(test_applicant)

print(f"Test applicant default probability: {prob:.2%}")
print(f"\nTop risk factors:")
print(risk_factors[['feature', 'shap_value', 'feature_value']].to_string(index=False))
print(f"\nTop mitigating factors:")
print(mitigating_factors[['feature', 'shap_value', 'feature_value']].to_string(index=False))

Test applicant default probability: 67.33%

Top risk factors:
             feature  shap_value  feature_value
        EXT_SOURCE_3    0.382547      -2.948272
      EXT_SOURCE_MIN    0.353216      -2.129759
     EXT_SOURCE_MEAN    0.266254      -0.548609
ANNUITY_INCOME_RATIO    0.083423       0.775865
       DAYS_EMPLOYED    0.065742       0.697319

Top mitigating factors:
                   feature  shap_value  feature_value
              EXT_SOURCE_1   -0.177847       1.230686
                 LOAN_TERM   -0.071664      -1.632247
              CNT_CHILDREN   -0.061359       2.192084
NAME_FAMILY_STATUS_Married   -0.033299       1.000000
            FLAG_OWN_CAR_N   -0.024912       0.000000


In [8]:
from groq import Groq

# Initialise Groq client
client = Groq(api_key=os.environ.get('GROQ_API_KEY'))

def generate_credit_summary(applicant_row, model, explainer):
    """
    Generates a plain English credit risk summary for a loan applicant
    using Groq (Llama 3) and SHAP explanations.
    """
    # Get prediction
    prob_default = model.predict_proba(applicant_row)[0][1]

    # Determine risk tier
    if prob_default < 0.3:
        risk_tier = "LOW RISK"
        recommendation = "APPROVE"
    elif prob_default < 0.6:
        risk_tier = "MEDIUM RISK"
        recommendation = "REVIEW"
    else:
        risk_tier = "HIGH RISK"
        recommendation = "DECLINE"

    # Get SHAP explanations
    risk_factors, mitigating_factors = get_applicant_shap_explanation(applicant_row)

    risk_text = "\n".join([
        f"- {row['feature'].replace('_', ' ')}: contribution = {row['shap_value']:.3f}"
        for _, row in risk_factors.iterrows()
    ])

    mitigating_text = "\n".join([
        f"- {row['feature'].replace('_', ' ')}: contribution = {row['shap_value']:.3f}"
        for _, row in mitigating_factors.iterrows()
    ])

    prompt = f"""You are a senior credit analyst at a fintech lending company.

APPLICANT ASSESSMENT:
- Predicted default probability: {prob_default:.2%}
- Risk tier: {risk_tier}
- Recommendation: {recommendation}

TOP RISK FACTORS:
{risk_text}

MITIGATING FACTORS:
{mitigating_text}

Write a professional credit risk summary of no more than 200 words.
State the risk tier and recommendation in the first sentence.
Explain the top 3 risk factors in plain English without technical jargon.
Acknowledge the mitigating factors.
End with a clear lending recommendation and next steps."""

    # Call Groq API
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        max_tokens=400,
        temperature=0.7,
        messages=[
            {
                "role": "system",
                "content": "You are a senior credit analyst at a fintech lending company. Write clear, professional credit risk summaries."
            },
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return {
        'probability': prob_default,
        'risk_tier': risk_tier,
        'recommendation': recommendation,
        'summary': response.choices[0].message.content
    }

print("Credit summary generator built successfully using Groq (Llama 3)")

Credit summary generator built successfully using Groq (Llama 3)


In [9]:
# Generate credit summary for our test applicant
print("Generating AI credit summary...")
print("This may take 30-60 seconds on first call — model loads on demand")
print("=" * 60)

result = generate_credit_summary(test_applicant, tuned_model, shap_explainer)

print(f"Default Probability: {result['probability']:.2%}")
print(f"Risk Tier:          {result['risk_tier']}")
print(f"Recommendation:     {result['recommendation']}")
print(f"\n--- AI GENERATED CREDIT SUMMARY ---\n")
print(result['summary'])
print("\n" + "=" * 60)

Generating AI credit summary...
This may take 30-60 seconds on first call — model loads on demand
Default Probability: 67.33%
Risk Tier:          HIGH RISK
Recommendation:     DECLINE

--- AI GENERATED CREDIT SUMMARY ---

**Credit Risk Summary**

This applicant is classified as HIGH RISK with a predicted default probability of 67.33%, and we recommend DECLINING the loan application.

The top risk factors contributing to this high risk assessment can be explained as follows: 

- **External credit sources** (EXT SOURCE 3, EXT SOURCE MIN, and EXT SOURCE MEAN) indicate that the applicant has a history of borrowing from external sources, which may indicate financial stress or a lack of access to traditional credit channels.
- **Annuity income ratio** (ANNUITY INCOME RATIO) suggests that a significant portion of the applicant's income is dedicated to debt repayment, leaving limited room for other expenses.
- **Days employed** (DAYS EMPLOYED) indicate a relatively short employment history, wh

In [10]:
# Generate summaries for all 10 demo applicants
print("Generating credit summaries for all demo applicants...")
print("=" * 60)

summaries = []

for i in range(len(X_demo)):
    applicant = X_demo.iloc[[i]]
    result = generate_credit_summary(applicant, tuned_model, shap_explainer)
    summaries.append(result)
    
    print(f"\nApplicant {i+1}")
    print(f"Probability: {result['probability']:.2%} | "
          f"Risk: {result['risk_tier']} | "
          f"Decision: {result['recommendation']}")

# Summary statistics
print("\n" + "=" * 60)
print("BATCH SUMMARY")
print("=" * 60)

decisions = [s['recommendation'] for s in summaries]
print(f"APPROVE:  {decisions.count('APPROVE')}")
print(f"REVIEW:   {decisions.count('REVIEW')}")
print(f"DECLINE:  {decisions.count('DECLINE')}")

avg_prob = np.mean([s['probability'] for s in summaries])
print(f"\nAverage default probability: {avg_prob:.2%}")
print(f"\nStep 6 complete — AI feature working end to end")

Generating credit summaries for all demo applicants...

Applicant 1
Probability: 67.33% | Risk: HIGH RISK | Decision: DECLINE

Applicant 2
Probability: 23.52% | Risk: LOW RISK | Decision: APPROVE

Applicant 3
Probability: 67.46% | Risk: HIGH RISK | Decision: DECLINE

Applicant 4
Probability: 14.28% | Risk: LOW RISK | Decision: APPROVE

Applicant 5
Probability: 46.23% | Risk: MEDIUM RISK | Decision: REVIEW

Applicant 6
Probability: 71.70% | Risk: HIGH RISK | Decision: DECLINE

Applicant 7
Probability: 22.89% | Risk: LOW RISK | Decision: APPROVE

Applicant 8
Probability: 78.24% | Risk: HIGH RISK | Decision: DECLINE

Applicant 9
Probability: 47.78% | Risk: MEDIUM RISK | Decision: REVIEW

Applicant 10
Probability: 26.85% | Risk: LOW RISK | Decision: APPROVE

BATCH SUMMARY
APPROVE:  4
REVIEW:   2
DECLINE:  4

Average default probability: 46.63%

Step 6 complete — AI feature working end to end


In [11]:
# Save summaries to a JSON file for use in deployment
import json

output = []
for i, s in enumerate(summaries):
    output.append({
        'applicant_id': i + 1,
        'probability': round(s['probability'], 4),
        'risk_tier': s['risk_tier'],
        'recommendation': s['recommendation'],
        'summary': s['summary']
    })

with open('../reports/sample_summaries.json', 'w') as f:
    json.dump(output, f, indent=2)

print("Sample summaries saved to reports/sample_summaries.json")
print("\n=== STEP 6 COMPLETE ===")
print("Generative AI feature:     WORKING")
print("Model:                     Llama 3.1 via Groq")
print("Input:                     Applicant data + SHAP values")
print("Output:                    Plain English credit risk summary")
print("Batch processing:          10 applicants tested successfully")
print("Deployment ready:          Yes — API key stored as env variable")

Sample summaries saved to reports/sample_summaries.json

=== STEP 6 COMPLETE ===
Generative AI feature:     WORKING
Model:                     Llama 3.1 via Groq
Input:                     Applicant data + SHAP values
Output:                    Plain English credit risk summary
Batch processing:          10 applicants tested successfully
Deployment ready:          Yes — API key stored as env variable
